In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from multiprocessing import Pool
import cv2

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## Test Data Processing

In [2]:
## for removing file if required
import shutil

try:
    shutil.rmtree("/kaggle/working/train")
    
    model_file_to_delete ="/kaggle/working/models"

    if os.path.isfile(model_file_to_delete):
        os.remove(model_file_to_delete)
    
except:
    print("No such directories")

No such directories


In [3]:
def load_data(data_dir):
    test_csv = os.path.join(data_dir, 'test.csv')
    
    test = pd.read_csv(test_csv)
    
    test_dir = os.path.join(data_dir, 'test_images/')
    
    test['file_path'] = test['id_code'].map(lambda x: os.path.join(test_dir, '{}.png'.format(x)))
    
    test['file_name'] = test["id_code"] + ".png"
        
    return test

In [4]:
data_dir = '/kaggle/input/aptos2019-blindness-detection/'
test_df = load_data(data_dir)

In [5]:
def crop_img(img, percentage):
    
    img_arr = np.array(img)
    img_gray = cv2.cvtColor(img_arr, cv2.COLOR_BGR2GRAY)
    
    threshold = img_gray > 0.1 * np.mean(img_gray[img_gray != 0])
    row_sums = np.sum(threshold, axis=1)
    col_sums = np.sum(threshold, axis=0)
    
    rows = np.where(row_sums > img_arr.shape[1] * percentage)[0]
    cols = np.where(col_sums > img_arr.shape[0] * percentage)[0]
    
    min_row, min_col = np.min(rows), np.min(cols)
    max_row, max_col = np.max(rows), np.max(cols)
    
    crop_img = img_arr[min_row : max_row + 1, min_col : max_col + 1]
    
    return Image.fromarray(crop_img)

In [6]:
def resize_maintain_aspect(img, desired_size):
    old_width, old_height = img.size
    aspect_ratio = old_width / old_height

    if aspect_ratio > 1:
        new_width = desired_size
        new_height = int(desired_size / aspect_ratio)
    else:
        new_height = desired_size
        new_width = int(desired_size * aspect_ratio)

    resized_img = img.resize((new_width, new_height), Image.ANTIALIAS)
    
    padded_image = Image.new("RGB", (desired_size, desired_size))
    x_offset = (desired_size - new_width) // 2
    y_offset = (desired_size - new_height) // 2
    padded_image.paste(resized_img, (x_offset, y_offset))
    
    return padded_image

In [7]:
def save_single(args):
    image_path, output_path_folder, percentage, output_size = args
    image = Image.open(image_path)
    
    # Display the image
    #plt.imshow(image)
    #plt.title('Original Image')
    #plt.show()
    
    croped_img = crop_img(image,percentage)
    image_resized = resize_maintain_aspect(croped_img, desired_size=output_size[0])
    
    #print(output_path_folder)
    #print(image_path)
    output_image_path = os.path.basename(image_path)
    # Save the resized image
    output_file_path = os.path.join(output_path_folder, output_image_path)
    #print(output_file_path)
    image_resized.save(output_file_path)

In [8]:
def fast_image_resize(df, output_path_folder, percentage, output_size=None):
    """Uses multiprocessing to make it fast"""
    if not output_size:
        warnings.warn("Need to specify output_size! For example: output_size=100")
        return

    if not os.path.exists(output_path_folder):
        os.makedirs(output_path_folder)
        
    jobs = []
    for df_item in range(len(df)):
        image_path = df.file_path.iloc[df_item]
        #print(image_path)
        job = (image_path, output_path_folder, percentage, output_size)
        jobs.append(job)
    
    """
    results = []
    for job in tqdm(jobs, total=len(jobs)):
        result = save_single(job)
        results.append(result)
    """
    with Pool() as p:
        list(tqdm(p.imap_unordered(save_single, jobs), total=len(jobs)))

In [9]:
percentage = 0.01
fast_image_resize(test_df, "/kaggle/working/test/images_resized/",percentage, output_size=(100, 100))

  0%|          | 0/367 [00:00<?, ?it/s]


AttributeError: module 'PIL.Image' has no attribute 'ANTIALIAS'

## Model Evaluation

In [10]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + '.png')
        image = Image.open(img_name)
        
        if self.transform:
            image = self.transform(image)
        
        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label

In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [12]:
test_csv_file = '/kaggle/input/aptos2019-blindness-detection/test.csv'
test_root_dir = '/kaggle/working/test/images_resized/'
test_dataset = BlindnessDataset(test_csv_file, test_root_dir, transform=transform, test=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [13]:
"""
model_paths = {
    'resnet18': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
    'efficientnet_b5': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/efficientnet_b5.pth",
    'inception_resnet_v2': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_resnet_v2.pth",
    'inception_v4': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_v4.pth",
    'seresnext50_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext50_32x4d.pth",
    'seresnext101_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext101_32x4d.pth"
}
"""
model_paths = {
    'inception_v4': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/3/inception_v4.pth",
}


model_names = {
    'resnet18': 'resnet18',
    'efficientnet_b5': 'efficientnet_b5',
    'inception_resnet_v2': 'inception_resnet_v2',
    'inception_v4': 'inception_v4',
    'seresnext50_32x4d': 'seresnext50_32x4d',
    'seresnext101_32x4d': 'seresnext101_32x4d'
}


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_list = []

# Initialize and load weights for each model using timm
for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    model = timm.create_model(model_name, pretrained=False, num_classes=5)
    model.load_state_dict(torch.load(path))
    model.to(device)
    model.eval()
    models_list.append(model)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/3/inception_v4.pth'

In [15]:
validation_scores = {
    #'resnet18': 0.887,
    #'efficientnet_b5': 0.952,
    #'inception_resnet_v2': 0.822,
    'inception_v4': 0.902 # 0.888,
    #'seresnext50_32x4d': 0.969 #0.709,
    #'seresnext101_32x4d': 0.951
}

In [16]:
total_score = sum(validation_scores.values())
weights = {k: v / total_score for k, v in validation_scores.items()}

In [17]:
all_outputs = []

with torch.no_grad():
    for images in tqdm(test_loader):
        images = images.to(device)
        outputs = [weights[model_key] * nn.functional.softmax(model(images), dim=1).unsqueeze(0) 
                   for model_key, model in zip(model_paths.keys(), models_list)]
        outputs = torch.cat(outputs)
        weighted_outputs = torch.sum(outputs, dim=0)
        all_outputs.extend(weighted_outputs.cpu().numpy())

all_outputs = np.array(all_outputs)
final_predictions = np.argmax(all_outputs, axis=1)

  0%|          | 0/23 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/test/images_resized/b460ca9fa26f.png'

In [18]:
submission_df = pd.DataFrame({
    'id_code': pd.read_csv(test_csv_file)['id_code'],
    'diagnosis': final_predictions
})

submission_df.to_csv('submission.csv', index=False)

NameError: name 'final_predictions' is not defined